In [ ]:
# 1. INSTALAÇÃO DAS BIBLIOTECAS
!pip install -q google-generativeai pandas

import pandas as pd
import io
import json
import re
from google.colab import files
import google.generativeai as genai

# A SUA CHAVE
from google.colab import userdata
MINHA_CHAVE_API = userdata.get('GEMINI_A')
genai.configure(api_key=MINHA_CHAVE_API)
modelo = genai.GenerativeModel('gemini-3.6-flash')

def limpar_dados():
    print("📂 Por favor, selecione o seu arquivo .csv (Covid, Futebol, Varejo, etc):")
    uploaded = files.upload()

    if not uploaded:
        print("Nenhum arquivo enviado.")
        return

    nome_arquivo = list(uploaded.keys())[0]
    print(f"\n⚙️ Lendo o arquivo '{nome_arquivo}' e detectando colunas automaticamente...")

    # Lê o arquivo
    df = pd.read_csv(io.BytesIO(uploaded[nome_arquivo]), sep=None, engine='python')
    amostra = df.head(3).to_string()

    prompt_escondido = f"""
    Você é um classificador de dados para engenharia de dados.
    Analise esta amostra de tabela:

    {amostra}

    Identifique o tipo semântico de CADA coluna. Responda APENAS com um arquivo JSON válido onde a chave é o nome exato da coluna e o valor é um destes 4 tipos:
    - "DATA" (datas)
    - "DINHEIRO" (valores financeiros, preços)
    - "NUMERO" (alturas, pesos, dimensões x y z, quantidades, etc)
    - "TEXTO" (texto geral)
    """

    print("🤖 O Agente IA está analisando a estrutura dos seus dados...")
    resposta_ia = modelo.generate_content(prompt_escondido)

    texto_json = resposta_ia.text.replace('```json', '').replace('```', '').strip()

    try:
        tipos_colunas = json.loads(texto_json)
    except json.JSONDecodeError:
        print("❌ Erro ao analisar a resposta da IA. Tente rodar novamente.")
        return

    print("🧹 Iniciando a limpeza pesada dos dados...")

    for coluna, tipo in tipos_colunas.items():
        if coluna not in df.columns:
            continue

        # REGRA DE OURO 1: Se a coluna já é número perfeito no original, a gente não estraga ela!
        if df[coluna].dtype in ['int64', 'float64'] and tipo in ["NUMERO", "DINHEIRO"]:
            continue

        if tipo == "DATA":
            df[coluna] = pd.to_datetime(df[coluna], errors='coerce').dt.date

        elif tipo == "DINHEIRO" and df[coluna].dtype == 'object':
            def limpar_moeda(val):
                if pd.isna(val): return val
                val = str(val).strip()
                val = re.sub(r'[^\d,.-]', '', val) # Limpa letras, mantém números
                if ',' in val and '.' in val:
                    if val.rfind(',') > val.rfind('.'): # BR: 1.500,50
                        val = val.replace('.', '').replace(',', '.')
                    else: # US: 1,500.50
                        val = val.replace(',', '')
                elif ',' in val:
                    val = val.replace(',', '.')
                return val

            df[coluna] = df[coluna].apply(limpar_moeda)
            df[coluna] = pd.to_numeric(df[coluna], errors='coerce')

        elif tipo == "NUMERO" and df[coluna].dtype == 'object':
            def extrair_numero(val):
                if pd.isna(val): return val
                val = str(val).replace(',', '.')
                match = re.search(r'([-+]?\d*\.?\d+|\d+)', val)
                return match.group(0) if match else val

            df[coluna] = df[coluna].apply(extrair_numero)
            df[coluna] = pd.to_numeric(df[coluna], errors='coerce')

    # REGRA DE OURO 2: Salvar traduzido para o Power BI Brasil!
    # (sep=';' para colunas e decimal=',' para números)
    novo_nome = "LIMPO_" + nome_arquivo
    df.to_csv(novo_nome, index=False, sep=';', decimal=',')

    print(f"✅ Sucesso! O arquivo '{novo_nome}' foi salvo perfeitamente no formato do Power BI Brasil (com vírgulas decimais).")
    files.download(novo_nome)

# Executa o programa
limpar_dados()

📂 Por favor, selecione o seu arquivo .csv (Covid, Futebol, Varejo, etc):


Saving train_and_test2.csv to train_and_test2.csv

⚙️ Lendo o arquivo 'train_and_test2.csv' e detectando colunas automaticamente...
🤖 O Agente IA está analisando a estrutura dos seus dados...
🧹 Iniciando a limpeza pesada dos dados...
✅ Sucesso! O arquivo 'LIMPO_train_and_test2.csv' foi salvo perfeitamente no formato do Power BI Brasil (com vírgulas decimais).


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!wget -O titanic.csv "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"

--2026-09-01 23:29:22--  https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 60302 (59K) [text/plain]
Saving to: ‘titanic.csv’

titanic.csv         100%[===================>]  58.89K  --.-KB/s    in 0.001s  

2026-09-01 23:29:23 (39.5 MB/s) - ‘titanic.csv’ saved [60302/60302]

